# 📝 RAG 파이프라인 과제 LV2(응용) — 청킹·자동 필터·출처 있는 답변

> LV1 에서 하나씩 익힌 것들을 **이어 붙입니다.** 문서를 잘라 색인하고, 청크에서 찾은 결과를 문서 단위로 되돌리고, 질문에서 분야를 **자동으로 뽑아** 필터로 걸고, 찾은 근거에 **출처를 붙여 답변을 생성**합니다.

## 풀이 방법
1. 위에서부터 **준비 셀**(제공 코드)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.

## 준비물
- 자료는 LV1 과 같은 `data/qna_docs.csv`(개인정보 질의응답 모음집 93건) 입니다.
- **3·4·7·8번은 실제 OpenAI 호출이 필요합니다.** 노트북이 있는 폴더에서 `cp .env.example .env` 로 복사한 뒤 본인 키를 채워 두세요. 이 노트북을 한 번 끝까지 돌리면 **약 15회** 호출됩니다(gpt-4o-mini 기준 몇 원 수준).

화이팅!

---
## 준비 — 지난 강의에서 만든 도구 되살리기
아래 셀들은 **실행만** 하세요. 이 강의에서 만든 청킹 함수·색인 도구·평가 지표를 그대로 가져옵니다.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀은 실행만 하세요.
# 15일차와 같은 방식입니다: .env 의 OPENAI_API_KEY 로 실제 OpenAI 에 연결합니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인한다 — OpenAI() 를 만든 뒤에 검사하면 SDK 인증 오류가 먼저 나서 이 안내가 묻힌다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 — OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from openai import OpenAI

client = OpenAI()
print("OpenAI 클라이언트 준비 완료 — 실제 API 연결됨")

In [ ]:
# [제공 코드] 질의응답 모음집을 불러옵니다
import pandas as pd

qna_docs = pd.read_csv('data/qna_docs.csv')
print(f'문서 수: {len(qna_docs)}')
display(qna_docs[['id', '분야', '조항', '쪽', '질문']].head(3))

In [ ]:
# [제공 코드] 임베딩 모델 준비 — 지난 시간에 쓴 한국어 문장 임베딩 모델입니다(불러오는 데 잠시 걸립니다).
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 (768차원)')

In [ ]:
# [제공 코드] 청킹 함수 — 이 강의 앞부분에서 만든 세 가지 청킹 전략입니다.
def chunk_fixed(text, size):
    """고정 크기(글자 수)로 자른다."""
    return [text[i:i + size] for i in range(0, len(text), size)]

def chunk_overlap(text, size, overlap):
    """앞 청크의 끝 일부를 다음 청크가 겹쳐 갖도록 자른다."""
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)]

def chunk_paragraph(text, size):
    """빈 줄로 나뉜 문단을 순서대로 모아 size 근처에서 끊는다.

    한 문단 자체는 절대 쪼개지 않는다 -- 문단이 size 보다 길면 그 조각 하나가
    size 를 그대로 넘어선 채로 들어간다."""
    chunks, cur = [], ''
    for para in [p.strip() for p in text.split('\n\n') if p.strip()]:
        if cur and len(cur) + len(para) > size:
            chunks.append(cur)
            cur = para
        else:
            cur = f'{cur}\n{para}' if cur else para
    if cur:
        chunks.append(cur)
    return chunks

In [ ]:
# [제공 코드] 색인·검색 도구 — 지난 시간(임베딩·벡터DB)에 배운 것을 함수로 묶어 둡니다.
import hashlib
from pathlib import Path

import chromadb

# 지난 시간에는 EphemeralClient(메모리)를 썼습니다. 오늘 문서는 수십 쪽이라 임베딩에 시간이 걸리니,
# PersistentClient 로 **디스크에 저장**합니다. 한 번 만들어 두면 커널을 새로 켜도 그대로 남아 있어
# 다시 임베딩하지 않습니다. (output/ 폴더는 실행 산출물이라 저장소에 올라가지 않습니다.)
CHROMA_DIR = Path('output' if Path('data').exists() else '../output') / 'chroma'
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def index_fingerprint(ids, texts, metadatas):
    """색인에 담긴 내용을 한 줄로 요약한 지문. 무엇 하나라도 바뀌면 값이 달라진다.

    메타데이터까지 넣는 이유: 본문이 그대로여도 메타에 열이 하나 늘면(예: 쪽 번호) 낡은
    색인에는 그 열이 없다. 그걸 모르고 다시 쓰면 검색은 되는데 meta['article'] 에서 KeyError 가 난다.
    """
    parts = ['\n'.join(ids), '\n'.join(texts),
             '\n'.join(repr(sorted(m.items())) for m in metadatas)]
    return hashlib.sha256('\x00'.join(parts).encode()).hexdigest()[:16]

def make_index(ids, texts, metadatas, name):
    """청크를 임베딩해 컬렉션으로 만든다. 같은 내용으로 이미 만들어 뒀으면 그대로 다시 쓴다."""
    want = index_fingerprint(ids, texts, metadatas)

    got = chroma.get_or_create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})
    if got.count() == len(ids) and (got.metadata or {}).get('fp') == want:
        print(f'{name}: 만들어 둔 색인을 그대로 씁니다 (청크 {got.count()}개)')
        return got

    # 개수나 지문이 다르면 문서가 바뀐 것이다 — 낡은 색인을 지우고 새로 만든다.
    chroma.delete_collection(name)
    col = chroma.create_collection(name, metadata={'hnsw:space': 'cosine', 'fp': want})

    emb = embed_model.encode(texts, normalize_embeddings=True)
    col.add(ids=ids, embeddings=emb.tolist(), documents=texts, metadatas=metadatas)

    print(f'{name}: 색인을 새로 만들었습니다 (청크 {col.count()}개)')
    return col

def search(col, query, k):
    """질문과 가장 가까운 청크 k개의 본문을 돌려준다."""
    qe = embed_model.encode([query], normalize_embeddings=True)
    res = col.query(query_embeddings=qe.tolist(), n_results=k)
    return res['documents'][0]

In [ ]:
# [제공 코드] 검색 품질 지표 — 지난 강의(평가)에서 손으로 구현한 네 가지 지표입니다.
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서가 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0

def precision_at_k(predicted, relevant, k):
    """상위 k개 중 관련 문서의 비율(관련 수 / k)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / k

def recall_at_k(predicted, relevant, k):
    """전체 관련 문서 중 상위 k개가 찾아낸 비율(관련 수 / 전체 관련 수)."""
    hits = sum(1 for p in predicted[:k] if p in relevant)
    return hits / len(relevant)

def mrr(predicted, relevant):
    """첫 번째 관련 문서 순위의 역수(1위면 1, 2위면 1/2 …). 없으면 0."""
    for i, p in enumerate(predicted, 1):
        if p in relevant:
            return 1 / i
    return 0.0

## 1. 문단 단위로 잘라 색인 만들기
**배경**: LV1 에서는 문서를 통째로 색인했습니다. 실무 문서는 한 건이 길어서 통째로 넣으면 질문과 상관없는 대목까지 함께 딸려 옵니다. 그래서 **문단 경계로 잘라** 조각을 색인합니다. 이때 조각마다 **어느 문서에서 나왔는지**를 메타데이터에 적어 둬야 나중에 문서로 되돌릴 수 있습니다.

**요구사항**: 세 리스트를 만들어 `make_index` 에 넘기세요.
- 각 문서의 `본문` 을 `chunk_paragraph(본문, 400)` 로 자릅니다.
- 조각 id 는 **`f'{문서 id}-{조각 번호}'`** 형식입니다(조각 번호는 문서 안에서 **0부터**). → 리스트 **`chunk_ids`**
- 조각 본문 → 리스트 **`chunk_texts`**
- 조각 메타데이터 → 리스트 **`chunk_metas`**. 키는 **`'doc_id'`·`'field'`(분야)·`'article'`(조항)·`'page'`(쪽, 정수)** 입니다.
- `make_index(chunk_ids, chunk_texts, chunk_metas, 'qna_lv2_chunks')` 로 만들어 변수 **`chunk_col`** 에 담으세요.

**예시**: 조각은 모두 **203개**가 나옵니다. 첫 조각의 id 는 `'q1-0'`, 메타데이터는 `{'doc_id': 'q1', 'field': '정의', 'article': '§2', 'page': 13}` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 문서를 하나씩 돌며 조각으로 자르고, 조각마다 세 리스트에 나란히 한 칸씩 채운다.

세부구현:
1. 빈 리스트 세 개를 만든다.
2. 표를 행 단위로 돌며 그 행의 본문을 문단 청킹한다.
3. 조각을 번호와 함께 돌며(0부터) id 문자열을 만들고 세 리스트에 각각 넣는다.
   3-1. 메타데이터의 쪽 값은 정수로 바꿔 넣는다.
4. 세 리스트와 컬렉션 이름을 make_index 에 넘겨 결과를 변수에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(chunk_ids) == 203, f'조각이 {len(chunk_ids)}개입니다 — chunk_paragraph 에 400 을 넘겼는지 확인하세요'
assert len(chunk_texts) == len(chunk_ids) == len(chunk_metas)
assert chunk_col.count() == 203
# q1 은 짧아 조각이 하나뿐이고 q2 는 둘로 갈린다 — 번호가 문서 안에서 0부터인지 본다
assert chunk_ids[:3] == ['q1-0', 'q2-0', 'q2-1']
# 잘라 넣었는지 — 조각은 원문보다 짧아야 하고, 원문 안에 들어 있어야 한다
assert max(len(t) for t in chunk_texts) < max(len(t) for t in qna_docs['본문'])
# 메타에 네 키가 다 있고 쪽이 정수인지 — 하나라도 빠지면 출처를 붙일 수 없다
assert set(chunk_metas[0]) == {'doc_id', 'field', 'article', 'page'}
assert all(isinstance(mm['page'], int) for mm in chunk_metas)
# id 의 앞부분이 그 조각의 doc_id 와 맞는지 — 문서로 되돌릴 때 쓰는 연결고리다
assert all(cid.rsplit('-', 1)[0] == mm['doc_id']
           for cid, mm in zip(chunk_ids, chunk_metas))
# 색인이 실제로 이 조각들로 만들어졌는지 검색해 확인한다
probe = chunk_col.query(query_embeddings=embed_model.encode(
    ['클라우드 서비스를 쓰면 그 회사가 수탁자가 되나요'], normalize_embeddings=True).tolist(), n_results=1)
assert probe['metadatas'][0][0]['doc_id'] == 'q90'
print('✅ 통과!')

## 2. 조각 검색 결과를 문서 단위로 되돌리기
**배경**: 조각을 색인했으니 검색도 조각으로 나옵니다. 그런데 상위 5조각이 **모두 같은 문서**에서 나오는 일이 흔합니다. 그대로 "관련 문서 5건" 이라고 세면 사실은 1건인데 5건으로 착각합니다. 그래서 조각을 **문서 단위로 접어** 세는 함수를 만듭니다.

**요구사항**:
- 함수 **`docs_of(col, query, k)`** 를 만드세요.
  1. `query` 를 임베딩해 `col.query(..., n_results=k * 5)` 로 **넉넉히** 조각을 가져옵니다.
  2. 결과 메타데이터를 **순서대로** 훑으며 `doc_id` 를 모으되, **이미 나온 문서는 건너뜁니다**.
  3. 서로 다른 문서가 `k` 개 모이면 멈추고, 그 리스트를 돌려줍니다.
- 만든 함수로 `'클라우드 서비스를 쓰면 그 회사가 수탁자가 되나요'` 를 **k=3** 으로 검색해 리스트 **`cloud_docs`** 에 담으세요.

**예시**: `cloud_docs` 는 서로 다른 문서 id 3개이고, 클라우드 수탁자 질의응답 **`'q90'`** 이 **첫 번째**입니다.

> 조각을 `k` 개만 가져오면 한 문서에서 다 나와 버려 문서가 `k` 개 안 모입니다. 그래서 넉넉히 가져와 접는 것입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 순서를 지키면서 중복만 걸러야 한다. 이미 담은 문서 id 인지 확인하며 앞에서부터 채운다.

세부구현:
1. 질의를 임베딩해 조각을 k 의 다섯 배만큼 조회한다.
2. 빈 리스트를 만들고 결과 메타데이터를 순서대로 훑는다.
   2-1. 그 조각의 문서 id 가 아직 리스트에 없으면 덧붙인다.
   2-2. 리스트 길이가 k 에 닿으면 반복을 멈춘다.
3. 리스트를 반환한다.
4. 만든 함수를 문제의 질의로 호출해 결과를 변수에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(cloud_docs) == 3 and len(set(cloud_docs)) == 3, '문서가 중복되었습니다'
assert cloud_docs[0] == 'q90'
# 조각 id 가 아니라 문서 id 인지 — 'q90-0' 처럼 번호가 붙어 있으면 접지 않은 것이다
assert all('-' not in d for d in cloud_docs)
# 지문에 없던 다른 질의로 다시 호출한다 — 한 질의에만 맞춘 답은 여기서 걸린다
other = docs_of(chunk_col, 'CCTV 안내판은 카메라마다 하나씩 붙여야 하나요', 3)
assert len(other) == 3 and len(set(other)) == 3
assert other[0] == 'q16', f'안내판 질의응답이 1위가 아닙니다: {other}'
# k 를 바꾸면 개수도 따라 바뀌는지 — k 를 무시하고 3으로 고정했으면 걸린다
assert len(docs_of(chunk_col, 'CCTV 안내판은 카메라마다 하나씩 붙여야 하나요', 5)) == 5
assert len(docs_of(chunk_col, 'CCTV 안내판은 카메라마다 하나씩 붙여야 하나요', 1)) == 1
print('✅ 통과!')

## 3. 질문에서 분야를 자동으로 뽑기
**배경**: LV1 7번에서는 사람이 `'영상정보'` 라고 직접 적어 필터를 걸었습니다. 실제 서비스에서 사용자는 질문만 던집니다. 그래서 **질문을 읽고 분야를 골라 주는 일**을 LLM 에게 맡깁니다. 이때 답을 자유 문장으로 받으면 매번 모양이 달라 필터에 쓸 수 없으므로, **정해진 틀(스키마)로** 받습니다.

**요구사항**:
- `pydantic` 의 `BaseModel` 을 상속한 스키마 **`FieldFilter`** 를 만드세요. 필드는 **`field`** 하나이고, 자료형은 `typing` 의 **`Literal`** 로 **여덟 분야 + `'없음'`** 만 허용하도록 못박으세요(모델이 아무 말이나 못 넣게).
- 함수 **`extract_field(query)`** 를 만드세요.
  1. `client.chat.completions.parse(model='gpt-4o-mini', temperature=0, messages=..., response_format=FieldFilter)` 로 호출합니다.
  2. system 메시지에도 **아래 여덟 분야를 후보로 알려 주고**, 어디에도 해당하지 않으면 `'없음'` 이라고 답하게 하세요.
  3. `resp.choices[0].message.parsed.field` 로 값을 꺼내, **여덟 분야 중 하나면 그 문자열**을, 아니면 **`None`** 을 돌려줍니다(`Literal` 로 막아 두어도 `'없음'` 은 오므로 이 걸러 내기가 필요합니다).

분야 여덟 가지(표의 `분야` 열과 글자까지 똑같아야 필터가 걸립니다):

```text
정의 · 영상정보 · 가명정보 · 공공서비스 · 민간사업자 · 민감·고유식별정보 · 위·수탁 · 기타 특수분야
```

**예시**: `extract_field('지문·얼굴 같은 민감정보를 근태관리에 써도 되나요')` → `'민감·고유식별정보'` / `extract_field('오늘 날씨 어때요')` → `None`

<details><summary>힌트</summary>

```text
접근방법:
- 받고 싶은 응답 구조를 먼저 클래스로 선언하고, 그 클래스를 응답 형식으로 넘긴다.
- 자료형으로 허용값을 못박아 두면 엉뚱한 문자열이 애초에 못 들어온다.
- 그래도 후보 안에 있는 없음 은 오므로, 받은 값을 목록과 대조해 걸러 낸다.

세부구현:
1. 후보 여덟 개와 없음 을 자료형으로 못박아 BaseModel 클래스의 필드로 선언한다.
2. 후보 여덟 개를 담은 리스트를 만들어 system 메시지 문장에 이어 붙인다.
3. 사용자 메시지로 질문을 넣고 응답 형식에 스키마를 지정해 호출한다.
4. 파싱된 값이 후보 목록에 있으면 그대로, 없으면 None 을 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert extract_field('지문·얼굴 같은 민감정보를 근태관리에 써도 되나요') == '민감·고유식별정보'
assert extract_field('오늘 날씨 어때요') is None, '후보 밖의 답은 None 으로 걸러야 합니다'
# 지문에 없던 다른 질문으로 다시 호출한다 — 예시 두 개만 맞춘 답은 여기서 걸린다
picked = extract_field('아파트 주차장에 CCTV 를 설치하려는데 안내판을 붙여야 하나요')
assert picked == '영상정보', f'CCTV 질문의 분야가 {picked} 로 나왔습니다'
assert extract_field('가명정보를 동의 없이 활용해도 되나요') == '가명정보'
# 뽑힌 값이 표의 분야 값과 글자까지 같아야 where 필터가 걸린다
assert picked in set(qna_docs['분야'])
# 스키마가 허용값을 못박고 있는지 — Literal 대신 str 로 두면 후보 밖 문자열이 그냥 들어온다
import typing
allowed = set(typing.get_args(FieldFilter.model_fields['field'].annotation))
assert set(qna_docs['분야']) <= allowed, \
    'FieldFilter 의 field 를 Literal 로 허용값을 못박았는지 확인하세요'
print('✅ 통과!')

## 4. 뽑은 분야를 필터로 걸어 검색하기
**배경**: 이제 3번이 고른 분야를 그대로 검색 조건으로 넘깁니다. 질문이 두루뭉술할수록 엉뚱한 분야의 문서가 위로 올라오는데, 분야를 좁히면 그런 잡음이 걸러집니다.

**요구사항**:
- 함수 **`search_smart(query, k)`** 를 만드세요.
  1. `extract_field(query)` 로 분야를 뽑습니다.
  2. 분야가 있으면 `col.query(..., where={'field': 그 분야})` 로, **`None` 이면 조건 없이** `chunk_col` 에서 조각을 `k * 5` 개 가져옵니다.
  3. 2번의 `docs_of` 와 같은 방식으로 **서로 다른 문서 `k` 개**를 돌려줍니다.
- `'지문·얼굴 같은 민감정보를 근태관리에 써도 되나요'` 를 **k=3** 으로 두 가지 방법으로 검색해 비교하세요.
  - `docs_of(chunk_col, ...)`(필터 없음) → **`plain_docs`**
  - `search_smart(...)`(필터 자동) → **`smart_docs`**
- 두 목록을 함께 출력해 무엇이 달라졌는지 보세요.

**예시**: `smart_docs` 는 **모두 `민감·고유식별정보` 분야** 문서이고, `plain_docs` 에는 다른 분야가 섞여 있습니다. 두 목록의 첫 번째는 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 조건이 있을 때와 없을 때 조회 방식만 다르고, 문서로 접는 과정은 같다.

세부구현:
1. 질문에서 분야를 뽑는다.
2. 질문을 임베딩한다.
3. 분야가 있으면 메타데이터 조건을 준 채로, 없으면 조건 없이 조각을 k 의 다섯 배 조회한다.
4. 결과 메타데이터를 순서대로 훑어 중복 없이 문서 id 를 k 개 모아 반환한다.
5. 같은 질문을 필터 없이도 검색해 두 목록을 나란히 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(plain_docs) == 3 and len(smart_docs) == 3
field_of = qna_docs.set_index('id')['분야'].to_dict()
assert all(field_of[d] == '민감·고유식별정보' for d in smart_docs), \
    'smart_docs 에 다른 분야가 섞였습니다 — where 조건이 걸렸는지 확인하세요'
assert not all(field_of[d] == '민감·고유식별정보' for d in plain_docs), \
    'plain_docs 는 필터 없이 검색한 결과여야 합니다'
assert plain_docs[0] == smart_docs[0] == 'q77'
# 분야를 못 고르는 질문에서도 결과가 나와야 한다 — None 일 때 조건을 걸면 0건이 된다
assert len(search_smart('오늘 날씨 어때요', 3)) == 3, \
    '분야가 None 이면 조건 없이 검색해야 합니다'
# 다른 질문으로 다시 호출해, 뽑힌 분야를 그대로 필터에 넘기는지 본다
cctv_query = '아파트 주차장에 CCTV 를 설치하려는데 안내판을 붙여야 하나요'
cctv_field = extract_field(cctv_query)
cctv_docs = search_smart(cctv_query, 3)
assert all(field_of[d] == cctv_field for d in cctv_docs), \
    f'뽑은 분야({cctv_field})와 다른 문서가 나왔습니다: {cctv_docs}'
print('✅ 통과!')

## 5. 평가셋으로 파이프라인 재기
**배경**: 질문 하나로는 검색이 좋은지 알 수 없습니다. **질문과 정답 문서를 짝지어 둔 평가셋**으로 여러 질문을 한꺼번에 재고, 그 **평균**을 봅니다. 준비 셀의 `hit_at_k`·`precision_at_k`·`recall_at_k`·`mrr` 을 씁니다.

아래 준비 셀이 질문 4개짜리 작은 평가셋 `mini_eval` 을 제공합니다(`(질문, 정답 문서 id 리스트)` 의 리스트).

**요구사항**:
- `mini_eval` 의 각 질문을 **`docs_of(chunk_col, 질문, 3)`** 으로 검색하세요(필터 없이).
- 네 지표를 **K=3** 으로 계산하고, 질문 4개의 **평균**을 딕셔너리 **`avg_scores`** 에 담으세요. 키는 **`'hit'`·`'precision'`·`'recall'`·`'mrr'`** 입니다.
- 질문별 결과도 한 줄씩 출력해 어떤 질문이 잘·못 되는지 보세요.

**예시**: `avg_scores` 는 `{'hit': 1.0, 'precision': ..., 'recall': ..., 'mrr': ...}` 처럼 네 개의 평균값을 담은 딕셔너리입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 질문마다 네 값을 구해 네 개의 리스트에 모으고, 마지막에 각각 평균을 낸다.

세부구현:
1. 지표별로 빈 리스트를 만든다.
2. 평가셋을 돌며 질문과 정답 목록을 꺼낸다.
   2-1. 그 질문으로 문서 3개를 검색한다.
   2-2. 네 지표를 각각 계산해 리스트에 넣는다(MRR 만 K 를 받지 않는다).
3. 각 리스트의 합을 개수로 나눠 딕셔너리에 담는다.
```

</details>

In [ ]:
# [제공 코드] 작은 평가셋 — 사람이 자료를 읽고 정답 문서를 손으로 붙인 것입니다
mini_eval = [
    ('아파트 주차장 CCTV 에 안내판을 몇 개나 붙여야 하나요', ['q16']),
    ('CCTV 로 찍은 영상은 며칠이나 보관할 수 있나요', ['q22', 'q14']),
    ('탈퇴한 회원의 회원번호만 남겨 둬도 되나요', ['q64']),
    ('클라우드 서비스를 쓰면 그 회사가 수탁자가 되나요', ['q90']),
]
print(f'평가 질문 수: {len(mini_eval)}')
for question, relevant in mini_eval:
    print(f'  {relevant} <- {question}')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert set(avg_scores) == {'hit', 'precision', 'recall', 'mrr'}
# 손으로 적은 값이 아니라 검색·지표를 실제로 돌린 값인지 다시 계산해 대조한다
recheck = []
for question, relevant in mini_eval:
    recheck.append(precision_at_k(docs_of(chunk_col, question, 3), relevant, 3))
assert abs(avg_scores['precision'] - sum(recheck) / len(recheck)) < 1e-9, \
    'precision 평균이 실제 검색 결과와 다릅니다'
# 네 질문 모두 상위 3에 정답이 있으므로 hit 평균은 1.0 이다
assert avg_scores['hit'] == 1.0
# 정답이 1~2개인데 3개를 가져오므로 정밀도는 1 이 될 수 없다
assert 0.2 < avg_scores['precision'] < 0.7
assert avg_scores['recall'] == 1.0 and avg_scores['mrr'] == 1.0
print('✅ 통과!')

## 6. 근거를 앞뒤로 넓히기
**배경**: 5번까지로 "어느 문서를 찾았나"는 잴 수 있게 됐습니다. 이제 그 문서로 답을 만들 차례인데, 그전에 근거를 손봐야 합니다. 조각으로 자르면 검색은 정확해지지만, 찾은 조각 **하나만** 읽으면 앞 문장이 없어 말이 끊깁니다("그렇지 않습니다" 만 남는 식이에요). 찾은 조각의 **바로 앞뒤 조각**을 함께 가져오면 문맥이 살아납니다. 다만 **다른 문서로 넘어가면 안 됩니다** — 옆 문서는 아예 다른 사례니까요.

**요구사항**:
- 함수 **`widen(chunk_id)`** 를 만드세요.
  - 조각 id 는 `'q70-1'` 처럼 **문서 id + `-` + 조각 번호** 형식입니다.
  - 그 조각과 **같은 문서의** 바로 앞(번호 −1)·바로 뒤(번호 +1) 조각 id 를 **번호 순서대로** 담은 리스트를 돌려줍니다.
  - 그 문서에 **없는 번호는 넣지 않습니다**(첫 조각이면 앞이, 마지막 조각이면 뒤가 없습니다). 실제로 있는 조각인지는 1번의 `chunk_ids` 로 확인하세요.
- 만든 함수로 `'q70-1'` 을 넓힌 결과를 리스트 **`widened`** 에 담으세요.
- 넓히기 **전과 후의 글자 수**를 각각 변수 **`narrow_len`**·**`wide_len`** 에 담아 함께 출력하세요(조각 본문은 `chunk_texts` 에 `chunk_ids` 와 같은 순서로 들어 있습니다).

**예시**: `widen('q70-1')` → `['q70-0', 'q70-1', 'q70-2']` · `widen('q1-0')` → `['q1-0']` (`q1` 은 조각이 하나뿐입니다). `wide_len` 은 `narrow_len` 보다 큽니다.

<details><summary>힌트</summary>

```text
접근방법:
- id 를 문서 부분과 번호 부분으로 나눈 뒤, 번호를 하나 빼고 하나 더한 후보 셋을 만든다.
- 후보 중 실제로 존재하는 것만 남기면 문서 경계와 양끝이 저절로 처리된다.

세부구현:
1. 오른쪽 구분자 하나를 기준으로 id 를 나누고 번호를 정수로 바꾼다.
2. 번호 −1·그대로·+1 로 후보 id 세 개를 만든다(문서 부분은 그대로 둔다).
3. 1번에서 만든 전체 조각 id 목록에 실제로 있는 것만 순서대로 남겨 반환한다.
4. 조각 id 로 본문을 찾을 수 있게 id 와 본문을 짝지어 두고, 길이를 재어 비교한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert widen('q70-1') == ['q70-0', 'q70-1', 'q70-2']
assert widen('q1-0') == ['q1-0'], 'q1 은 조각이 하나뿐입니다'
assert widened == widen('q70-1')
# 양끝 — 없는 번호를 만들어 내면 안 된다(q2 는 조각이 둘뿐)
assert widen('q2-0') == ['q2-0', 'q2-1']
assert widen('q2-1') == ['q2-0', 'q2-1']
# 문서 경계를 넘지 않는다 — q2 의 마지막 조각을 넓혀도 q3 는 들어오면 안 된다
assert all(c.rsplit('-', 1)[0] == 'q2' for c in widen('q2-1'))
# 넓히면 근거가 실제로 길어진다(그게 목적이다)
assert wide_len > narrow_len
check_of = dict(zip(chunk_ids, chunk_texts))
assert wide_len == sum(len(check_of[c]) for c in widen('q70-1'))
assert narrow_len == len(check_of['q70-1'])
print('✅ 통과!')

## 7. 넓힌 근거로 출처 붙인 답변 만들기
**배경**: 6번에서 넓힌 근거를 이제 실제 답변에 씁니다. 사용자는 문서 id 가 아니라 **답변**을 원하니까요. 다만 그냥 답하게 두면 모델이 자료에 없는 말을 지어냅니다. 그래서 **찾은 근거만 주고**, 답 끝에 **어디서 왔는지 적게** 합니다. 근거가 어느 분야 몇 쪽인지는 1번에서 메타데이터에 실어 뒀습니다.

**요구사항**:
- 함수 **`answer_with_source(query)`** 를 만드세요. 반환값은 **`(답변 문자열, 근거로 쓴 조각 id 목록)`** 두 개입니다.
  1. `chunk_col` 에서 질문과 가까운 **조각 3개**를 가져옵니다(`n_results=3`).
  2. 그 3개를 **각각 6번의 `widen` 으로 넓혀**, 나온 조각 id 를 **중복 없이 순서대로** 모읍니다 → 이 조각 id 목록이 반환할 두 번째 값입니다.
  3. 모은 조각들의 본문과 출처(`분야`·`조항`·`쪽`)를 이어 붙여 근거 문자열을 만듭니다.
  4. `client.chat.completions.create(model='gpt-4o-mini', temperature=0, max_tokens=400, ...)` 로 답을 만듭니다. 규칙을 **system 메시지에 분명히** 적으세요.
     - 주어진 근거만 사용한다
     - 답의 마지막 줄에 **`(출처: 분야 조항, N쪽)`** 형식으로 쓴 근거를 밝힌다
     - 근거에서 답을 찾을 수 없으면 **`자료에서 찾지 못했습니다`** 라고만 답한다
  5. 답변 문자열과 조각 id 목록을 함께 돌려줍니다.
- 만든 함수로 `'수탁 업체가 개인정보를 잘 다루는지 어떻게 감독하나요'` 을 물어 **`sign_answer`** 와 **`evidence_ids`** 에 나눠 담고 둘 다 출력하세요.

**예시**: `sign_answer` 는 답변 본문 뒤에 `(출처: 위·수탁 §26, 111쪽)` 같은 줄이 붙은 문자열이고, `evidence_ids` 는 `['q86-0', 'q86-1', ...]` 처럼 **3개보다 많은** 조각 id 목록입니다(찾은 3개를 넓혔으니까요).

<details><summary>힌트</summary>

```text
접근방법:
- 근거 하나마다 그 출처를 바로 앞줄에 적어 함께 넘긴다.
- 규칙은 사람에게 설명하듯 구체적으로 적고, 원하는 출처 형식은 예시를 하나 함께 보여 준다.

세부구현:
1. 질문을 임베딩해 조각 3개를 조회한다(조각 id 도 함께 받는다).
2. 조각 id 를 하나씩 넓히고, 아직 담지 않은 것만 순서대로 모아 근거 id 목록을 만든다.
3. 그 id 로 본문과 메타데이터를 찾아 '출처 표기 + 본문' 을 한 묶음씩 만들어 하나로 잇는다.
   3-1. id 로 본문·메타데이터를 찾으려면 1번의 세 리스트를 짝지어 두면 편하다.
4. system 메시지에 세 규칙을 적고, user 메시지에 근거와 질문을 함께 넣는다.
5. 응답의 첫 선택지에서 내용 문자열을 꺼내 근거 id 목록과 함께 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import re

assert isinstance(sign_answer, str) and len(sign_answer) > 30
assert '(출처:' in sign_answer, '답의 마지막에 출처 표기가 없습니다'
assert '자료에서 찾지 못했습니다' not in sign_answer, '자료에 있는 질문인데 못 찾았다고 답했습니다'

# 답에 적힌 쪽 번호가 '실제로 검색된 조각의 쪽'인지 확인한다 — 지어낸 출처를 걸러 낸다
cited_pages = {int(n) for n in re.findall(r'(\d+)\s*쪽', sign_answer)}
assert cited_pages, '쪽 번호가 없습니다'
probe_vec = embed_model.encode(['수탁 업체가 개인정보를 잘 다루는지 어떻게 감독하나요'], normalize_embeddings=True)
probe = chunk_col.query(query_embeddings=probe_vec.tolist(), n_results=3)
real_pages = {mm['page'] for mm in probe['metadatas'][0]}
assert cited_pages <= real_pages, \
    f'검색되지 않은 쪽을 인용했습니다: {cited_pages - real_pages}'

# 근거가 6번 widen 으로 넓힌 결과와 같은지 — 넓히지 않으면 3개에서 멈춘다
want_ids = []
for chunk_id in probe['ids'][0]:
    for neighbour in widen(chunk_id):
        if neighbour not in want_ids:
            want_ids.append(neighbour)
assert evidence_ids == want_ids, \
    '근거 조각이 6번 widen 으로 넓힌 결과와 다릅니다'
assert len(evidence_ids) > 3, '앞뒤 조각까지 붙이면 3개보다 많아집니다'

# 지문에 없던 다른 질문으로 다시 호출한다 — 한 질문에만 맞춘 답은 여기서 걸린다
cloud_answer, cloud_ids = answer_with_source('클라우드 사업자가 수탁자에 해당하나요')
assert '(출처:' in cloud_answer
other_pages = {int(n) for n in re.findall(r'(\d+)\s*쪽', cloud_answer)}
assert other_pages and other_pages != cited_pages, \
    '질문이 달라졌는데 같은 출처가 나옵니다 — 질문마다 검색하고 있는지 확인하세요'
print('✅ 통과!')

## 8. 자료에 없는 질문에는 없다고 답하기
**배경**: RAG 에서 가장 위험한 실패는 "모른다"고 해야 할 때 그럴듯한 답을 지어내는 것입니다. 7번에서 규칙을 넣었으니 **정말로 듣는지** 확인해야 합니다. 규칙은 적어 놓는다고 지켜지는 게 아니라 **재 봐야 아는** 것이에요.

**요구사항**:
- 7번의 `answer_with_source` 를 그대로 써서 `'개인정보를 잘 지키면 세금을 깎아 주나요'` 을 물어 변수 **`none_answer`** 에 담고 출력하세요(이 자료에는 세금 이야기가 없습니다).
- 답에 **`자료에서 찾지 못했습니다`** 가 들어 있는지 눈으로 확인하세요(반환값이 둘이니 **`none_answer`·`none_ids`** 로 나눠 받으세요).

**예시**: `none_answer` 에는 `자료에서 찾지 못했습니다` 가 들어 있고, `(출처:` 는 없습니다.

> 만약 모델이 규칙을 어기고 답을 지어낸다면 7번의 system 메시지를 더 분명하게 고쳐야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 새로 만들 것은 없다. 7번 함수에 자료 밖의 질문을 넣어 결과를 본다.

세부구현:
1. 7번에서 만든 함수에 자료에 없는 질문을 넘겨 결과를 변수에 담는다.
2. 결과를 출력해 규칙대로 답했는지 확인한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(none_answer, str)
assert '자료에서 찾지 못했습니다' in none_answer, \
    '자료에 없는 질문인데 답을 지어냈습니다 — system 메시지의 규칙을 더 분명히 적으세요'
# 없다고 해 놓고 출처를 붙이면 앞뒤가 안 맞는다
assert '(출처:' not in none_answer
# 7번 답변과 다른 문자열인지 — 같은 값을 다시 담았으면 걸린다
assert none_answer != sign_answer
print('✅ 통과!')

---
수고했어요! LV2 에서 **청킹→색인→문서 단위 검색→분야 자동 추출→필터 검색→출처 있는 답변**까지 이어 붙였습니다.

LV3 에서는 이 파이프라인을 **평가셋으로 재고, 실패를 진단하고, 한 곳을 고쳐 다시 재는** 흐름을 끝까지 해 봅니다.